# Clean Nutrition DataFrames (OFF + USDA)

This notebook creates **two clean, ready-to-use DataFrames** from `/data/raw`:
- `off_nutrition_df` from `off.duckdb` (`nutrition_medium`)
- `usda_nutrition_df` from `usda.duckdb` (`usda_macros_clean`)

Both are normalized to the same schema and saved under `data/processed/`.

### Data origin:

USDA: https://fdc.nal.usda.gov/download-datasets

openfoodfacts: https://world.openfoodfacts.org/data

# Important Note:

## This notebook only works with the .duckdb files. These are not included in the github directory, because they exceed the maximum data size for uploads

In [12]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)


In [13]:
ROOT = Path.cwd().resolve()
RAW_DIR = ROOT / 'raw'
PROCESSED_DIR = ROOT / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OFF_DB = RAW_DIR / 'off.duckdb'
USDA_DB = RAW_DIR / 'usda.duckdb'

assert OFF_DB.exists(), f'Missing: {OFF_DB}'
assert USDA_DB.exists(), f'Missing: {USDA_DB}'

OFF_DB, USDA_DB


(PosixPath('/Users/kevin/Documents/bootcamp_ai/sayfit-capstone1/data/raw/off.duckdb'),
 PosixPath('/Users/kevin/Documents/bootcamp_ai/sayfit-capstone1/data/raw/usda.duckdb'))

## Source Column Explorer
Use these cells to inspect all available columns before deciding what to keep in the clean outputs.


In [14]:
def list_table_columns(db_path: Path, table_name: str) -> pd.DataFrame:
    with duckdb.connect(str(db_path), read_only=True) as con:
        return con.sql(f"DESCRIBE {table_name}").df()

# OFF: broad raw table + current nutrition table
off_products_cols = list_table_columns(OFF_DB, 'products')
off_nutrition_cols = list_table_columns(OFF_DB, 'nutrition_medium')

# USDA: current clean macro table + portion-related source table
usda_macros_cols = list_table_columns(USDA_DB, 'usda_macros_clean')
usda_food_portion_cols = list_table_columns(USDA_DB, 'food_portion')

print('OFF products columns:', len(off_products_cols))
display(off_products_cols)

print('OFF nutrition_medium columns:', len(off_nutrition_cols))
display(off_nutrition_cols)

print('USDA usda_macros_clean columns:', len(usda_macros_cols))
display(usda_macros_cols)

print('USDA food_portion columns:', len(usda_food_portion_cols))
display(usda_food_portion_cols)


OFF products columns: 209


,column_name,column_type,null,key,default,extra
0,code,VARCHAR,YES,None,None,None
1,url,VARCHAR,YES,None,None,None
2,creator,VARCHAR,YES,None,None,None
3,created_t,BIGINT,YES,None,None,None
4,created_datetime,TIMESTAMP WITH TIME ZONE,YES,None,None,None
...,...,...,...,...,...,...
204,carnitine_100g,VARCHAR,YES,None,None,None
205,sulphate_100g,VARCHAR,YES,None,None,None
206,nitrate_100g,VARCHAR,YES,None,None,None
207,acidity_100g,VARCHAR,YES,None,None,None


OFF nutrition_medium columns: 9


,column_name,column_type,null,key,default,extra
0,code,VARCHAR,YES,None,None,None
1,product_name,VARCHAR,YES,None,None,None
2,brands,VARCHAR,YES,None,None,None
3,quantity,VARCHAR,YES,None,None,None
4,serving_size,VARCHAR,YES,None,None,None
5,kcal_100g,DOUBLE,YES,None,None,None
6,protein_100g,DOUBLE,YES,None,None,None
7,carbs_100g,DOUBLE,YES,None,None,None
8,fat_100g,DOUBLE,YES,None,None,None


USDA usda_macros_clean columns: 6


,column_name,column_type,null,key,default,extra
0,fdc_id,BIGINT,YES,None,None,None
1,description,VARCHAR,YES,None,None,None
2,kcal_100g,DOUBLE,YES,None,None,None
3,protein_100g,DOUBLE,YES,None,None,None
4,carbs_100g,DOUBLE,YES,None,None,None
5,fat_100g,DOUBLE,YES,None,None,None


USDA food_portion columns: 11


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,fdc_id,VARCHAR,YES,None,None,None
2,seq_num,VARCHAR,YES,None,None,None
3,amount,VARCHAR,YES,None,None,None
4,measure_unit_id,VARCHAR,YES,None,None,None
5,portion_description,VARCHAR,YES,None,None,None
6,modifier,VARCHAR,YES,None,None,None
7,gram_weight,VARCHAR,YES,None,None,None
8,data_points,VARCHAR,YES,None,None,None
9,footnote,VARCHAR,YES,None,None,None


In [15]:
# Quick helper: find likely portion/serving columns by name pattern
portion_keywords = ['portion', 'serving', 'gram', 'weight', 'measure', 'unit', 'household']

def find_columns_by_keywords(cols_df: pd.DataFrame, keywords: list[str]) -> pd.DataFrame:
    pattern = '|'.join(keywords)
    mask = cols_df['column_name'].str.lower().str.contains(pattern, regex=True)
    return cols_df.loc[mask].reset_index(drop=True)

print('OFF products portion/serving candidates:')
display(find_columns_by_keywords(off_products_cols, portion_keywords))

print('USDA food_portion candidates:')
display(find_columns_by_keywords(usda_food_portion_cols, portion_keywords))


OFF products portion/serving candidates:


,column_name,column_type,null,key,default,extra
0,serving_size,VARCHAR,YES,None,None,None
1,serving_quantity,DOUBLE,YES,None,None,None


USDA food_portion candidates:


,column_name,column_type,null,key,default,extra
0,measure_unit_id,VARCHAR,YES,None,None,None
1,portion_description,VARCHAR,YES,None,None,None
2,gram_weight,VARCHAR,YES,None,None,None


In [22]:
def load_off_nutrition(db_path: Path) -> pd.DataFrame:
    query = '''
        SELECT
            TRIM(code) AS item_id,
            TRIM(product_name) AS item_name,
            NULLIF(TRIM(brands), '') AS brand,
            kcal_100g,
            protein_100g,
            carbs_100g,
            fat_100g,
            serving_size,
            quantity,
            'openfoodfacts' AS source
        FROM nutrition_medium
        WHERE code IS NOT NULL
          AND product_name IS NOT NULL
          AND TRIM(product_name) <> ''
          AND kcal_100g IS NOT NULL
          AND protein_100g IS NOT NULL
          AND carbs_100g IS NOT NULL
          AND fat_100g IS NOT NULL
          AND kcal_100g BETWEEN 0 AND 1000
          AND protein_100g BETWEEN 0 AND 100
          AND carbs_100g BETWEEN 0 AND 100
          AND fat_100g BETWEEN 0 AND 100
    '''
    with duckdb.connect(str(db_path), read_only=True) as con:
        df = con.sql(query).df()

    # OFF codes should be unique, keep first if duplicates appear.
    df = df.drop_duplicates(subset=['item_id']).reset_index(drop=True)
    return df


def load_usda_nutrition(db_path: Path) -> pd.DataFrame:
    query = '''
        WITH best_portion AS (
            SELECT
                fdc_id,
                NULLIF(TRIM(portion_description), '') AS portion_description,
                NULLIF(TRIM(modifier), '') AS modifier,
                TRY_CAST(gram_weight AS DOUBLE) AS gram_weight,
                ROW_NUMBER() OVER (
                    PARTITION BY fdc_id
                    ORDER BY
                        CASE WHEN TRY_CAST(gram_weight AS DOUBLE) IS NOT NULL THEN 0 ELSE 1 END,
                        TRY_CAST(gram_weight AS DOUBLE) DESC,
                        CASE WHEN NULLIF(TRIM(portion_description), '') IS NOT NULL THEN 0 ELSE 1 END,
                        CASE WHEN NULLIF(TRIM(modifier), '') IS NOT NULL THEN 0 ELSE 1 END,
                        TRY_CAST(seq_num AS INTEGER),
                        id
                ) AS rn
            FROM food_portion
            WHERE fdc_id IS NOT NULL
        )
        SELECT
            CAST(m.fdc_id AS VARCHAR) AS item_id,
            TRIM(m.description) AS item_name,
            CAST(NULL AS VARCHAR) AS brand,
            m.kcal_100g,
            m.protein_100g,
            m.carbs_100g,
            m.fat_100g,
            COALESCE(p.portion_description, p.modifier) AS portion_description,
            p.gram_weight,
            'usda' AS source
        FROM usda_macros_clean m
        LEFT JOIN best_portion p
          ON CAST(m.fdc_id AS VARCHAR) = p.fdc_id
         AND p.rn = 1
        WHERE m.fdc_id IS NOT NULL
          AND m.description IS NOT NULL
          AND TRIM(m.description) <> ''
          AND m.kcal_100g IS NOT NULL
          AND m.protein_100g IS NOT NULL
          AND m.carbs_100g IS NOT NULL
          AND m.fat_100g IS NOT NULL
          AND m.kcal_100g BETWEEN 0 AND 1000
          AND m.protein_100g BETWEEN 0 AND 100
          AND m.carbs_100g BETWEEN 0 AND 100
          AND m.fat_100g BETWEEN 0 AND 100
    '''
    with duckdb.connect(str(db_path), read_only=True) as con:
        df = con.sql(query).df()

    # FDC ids should be unique, keep first if duplicates appear.
    df = df.drop_duplicates(subset=['item_id']).reset_index(drop=True)
    return df


In [23]:
off_nutrition_df = load_off_nutrition(OFF_DB)
usda_nutrition_df = load_usda_nutrition(USDA_DB)

print('off_nutrition_df:', off_nutrition_df.shape)
print('usda_nutrition_df:', usda_nutrition_df.shape)


off_nutrition_df: (66341, 10)
usda_nutrition_df: (1817628, 10)


In [24]:
off_nutrition_df.head()


,item_id,item_name,brand,kcal_100g,protein_100g,carbs_100g,fat_100g,serving_size,quantity,source
0,0000103783001,greek yogurt,Chobani,86.666667,7.333333,10.666667,1.666667,NaN,NaN,openfoodfacts
1,0000107972030,Fajita Chicken W/ Wholegrain Rice,Fuelhub,97.227723,10.891089,10.297030,1.049505,1 portion (505 g),NaN,openfoodfacts
2,00001247,Shake Mix Vanilla Flavour,Herbalife,376.000000,36.000000,42.000000,7.200000,100.0g,NaN,openfoodfacts
3,0000127534563,German fine bread,NaN,263.000000,7.020000,52.630000,3.510000,2 SLICES (57 g),NaN,openfoodfacts
4,00001328,Harvest whole wheat bread,"Trader Joe's, E.Leclerc",232.558140,11.627907,41.860465,4.651163,NaN,NaN,openfoodfacts


In [25]:
usda_nutrition_df.head()


,item_id,item_name,brand,kcal_100g,protein_100g,carbs_100g,fat_100g,portion_description,gram_weight,source
0,1251280,PARMESAN PEPPERCORN DIP,None,188.0,12.50,6.25,15.62,NaN,NaN,usda
1,1251668,STAR WARS HOT COCOA MIX,None,393.0,0.00,96.43,0.00,NaN,NaN,usda
2,1251760,"GOOD MEADOW, CLASSIC COCONUT CAKE",None,395.0,2.63,59.21,17.11,NaN,NaN,usda
3,1251788,SOURDOUGH MULTIGRAIN,None,246.0,7.02,47.37,1.75,NaN,NaN,usda
4,1252048,"REDUCED SODIUM CHICKEN BROTH, RICH CHICKEN",None,6.0,1.25,0.42,0.00,NaN,NaN,usda


In [26]:
def quality_report(df: pd.DataFrame, name: str) -> None:
    print(f'\n{name}')
    print('-' * len(name))
    print('Rows:', len(df))
    print('Nulls by column:')
    print(df.isna().sum())


quality_report(off_nutrition_df, 'off_nutrition_df')
quality_report(usda_nutrition_df, 'usda_nutrition_df')



off_nutrition_df
----------------
Rows: 66341
Nulls by column:
item_id             0
item_name           0
brand            9560
kcal_100g           0
protein_100g        0
carbs_100g          0
fat_100g            0
serving_size    18630
quantity        38406
source              0
dtype: int64

usda_nutrition_df
-----------------
Rows: 1817628
Nulls by column:
item_id                      0
item_name                    0
brand                  1817628
kcal_100g                    0
protein_100g                 0
carbs_100g                   0
fat_100g                     0
portion_description    1804656
gram_weight            1804600
source                       0
dtype: int64


In [27]:
off_out = PROCESSED_DIR / 'off_nutrition_clean.parquet'
usda_out = PROCESSED_DIR / 'usda_nutrition_clean.parquet'

def save_df(df: pd.DataFrame, parquet_path: Path, csv_name: str) -> Path:
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception:
        csv_path = PROCESSED_DIR / csv_name
        df.to_csv(csv_path, index=False)
        return csv_path

off_saved = save_df(off_nutrition_df, off_out, 'off_nutrition_clean.csv')
usda_saved = save_df(usda_nutrition_df, usda_out, 'usda_nutrition_clean.csv')

print(f'Saved: {off_saved}')
print(f'Saved: {usda_saved}')


Saved: /Users/kevin/Documents/bootcamp_ai/sayfit-capstone1/data/processed/off_nutrition_clean.csv
Saved: /Users/kevin/Documents/bootcamp_ai/sayfit-capstone1/data/processed/usda_nutrition_clean.csv


## Final outputs

- `off_nutrition_df`
- `usda_nutrition_df`

Both are clean, schema-aligned, and persisted in `data/processed/`.
